# Fine-tuning DINOv2-large para Nose Print

Paw Friend — Refactor Maestro 2026-04-24

**Objetivo**: mejorar la separación same-pet vs cross-pet de 0.445 (baseline) a >0.80 para poder usar en producción.

**Método**: fine-tuning con triplet loss sobre DogFaceNet dataset (~8,677 imágenes, 1,393 perros).

**Requisitos**:
- Runtime: **T4 GPU** (gratis en Colab) — Runtime → Change runtime type → T4 GPU
- ~1 hora total
- Tus 49 fotos de test + los 3 datasets descargados en Drive o directo en Colab

**Instrucciones**: correr cada celda con Shift+Enter, en orden.

## 1. Setup — verificar GPU + instalar dependencies

In [ ]:
# Verificar GPU
!nvidia-smi

In [ ]:
# Instalar dependencies (Colab trae torch ya, solo agregamos transformers)
!pip install -q transformers pillow scikit-learn tqdm

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponible: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 2. Descargar datasets directo en Colab (más rápido que subir desde tu laptop)

In [ ]:
import os
os.makedirs('/content/data', exist_ok=True)

# DogFaceNet (el más relevante para re-ID)
!git clone --depth 1 https://github.com/GuillaumeMougeot/DogFaceNet.git /content/data/dogfacenet 2>/dev/null || echo 'Ya clonado'
print('\nContenido DogFaceNet:')
!ls /content/data/dogfacenet

In [ ]:
# Alternativa: Oxford Pets (si DogFaceNet no tiene data en el repo)
# Oxford Pets tiene bounding boxes de caras, sirve si queremos crop preciso
!wget -q https://www.robots.ox.ac.uk/~vgg/data/pets/data/images.tar.gz -O /content/oxford_images.tar.gz
!mkdir -p /content/data/oxford-pets
!tar -xzf /content/oxford_images.tar.gz -C /content/data/oxford-pets
print('\nContenido Oxford Pets (primeras 10):')
!ls /content/data/oxford-pets/images | head -10
print('\nTotal imagenes Oxford:')
!ls /content/data/oxford-pets/images | wc -l

## 3. Subir tus 49 fotos de test (eval set)

Hay 2 opciones:

**A. Upload directo**: `Files` panel izquierdo → upload → zipeá `_pending/nose_print_test_photos/` y subí el zip.

**B. Desde Google Drive** (más fácil si ya lo tenés allí):

In [ ]:
# Opción B — desde Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Copiar las 49 fotos a /content/test_photos/
# AJUSTAR LA RUTA SEGUN DONDE TENGAS EL ZIP EN TU DRIVE
import shutil, os

TEST_PHOTOS_SOURCE = '/content/drive/MyDrive/Paw Friend/nose_print_test_photos.zip'  # AJUSTAR

if os.path.exists(TEST_PHOTOS_SOURCE):
    !cp "{TEST_PHOTOS_SOURCE}" /content/test_photos.zip
    !unzip -oq /content/test_photos.zip -d /content/
    !mv /content/nose_print_test_photos /content/test_photos 2>/dev/null || true
    print('Fotos de test:')
    !ls /content/test_photos
else:
    print(f'No encontrado: {TEST_PHOTOS_SOURCE}')
    print('Ajusta la variable TEST_PHOTOS_SOURCE arriba.')

## 4. Cargar DINOv2-large (modelo base)

In [ ]:
from transformers import AutoImageProcessor, AutoModel

MODEL_ID = 'facebook/dinov2-large'
print(f'Cargando {MODEL_ID}...')
processor = AutoImageProcessor.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(MODEL_ID).to(device)
print(f'Modelo cargado. Embedding dim: {model.config.hidden_size}')
print(f'Parametros: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M')

## 5. Dataset + Dataloader para triplet loss

Triplet loss necesita: (anchor, positive, negative) donde:
- anchor + positive = **mismo perro**, fotos distintas
- negative = **otro perro**

El modelo aprende a hacer embedding(anchor) cercano a embedding(positive) y lejano de embedding(negative).

In [ ]:
import os
import glob
import random
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torch.nn.functional as F

# Usamos Oxford Pets: nombre de archivo encodea la raza (proxy de identidad)
# Para DogFaceNet si el dataset viene como .npy, requiere adaptación (ver al final del notebook)

OXFORD_DIR = '/content/data/oxford-pets/images'

def get_oxford_pets_grouped():
    """Oxford Pets: archivo 'breed_N.jpg'. Agrupamos por raza como proxy identidad.
    Para biometria real es menos ideal (misma raza != mismo perro) pero sirve
    para fine-tuning inicial."""
    files = sorted(glob.glob(f'{OXFORD_DIR}/*.jpg'))
    groups = {}
    for f in files:
        name = Path(f).stem
        # 'abyssinian_100' → 'abyssinian'
        breed = '_'.join(name.split('_')[:-1])
        if breed not in groups:
            groups[breed] = []
        groups[breed].append(f)
    # Filtrar razas con <3 fotos
    groups = {k: v for k, v in groups.items() if len(v) >= 3}
    print(f'Razas con >=3 fotos: {len(groups)}')
    print(f'Total imagenes: {sum(len(v) for v in groups.values())}')
    return groups

pet_groups = get_oxford_pets_grouped()

In [ ]:
# Transformaciones
IMAGE_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE + 16, IMAGE_SIZE + 16)),
    transforms.RandomCrop(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class TripletDataset(Dataset):
    def __init__(self, groups, transform):
        self.groups = groups
        self.breeds = list(groups.keys())
        self.transform = transform
        # Flatten para __len__
        self.all_samples = [(b, f) for b, files in groups.items() for f in files]

    def __len__(self):
        return len(self.all_samples)

    def __getitem__(self, idx):
        anchor_breed, anchor_path = self.all_samples[idx]

        # Positive: otra foto del mismo grupo
        same_group = [f for f in self.groups[anchor_breed] if f != anchor_path]
        pos_path = random.choice(same_group)

        # Negative: foto de otro grupo
        other_breed = random.choice([b for b in self.breeds if b != anchor_breed])
        neg_path = random.choice(self.groups[other_breed])

        try:
            anchor = self.transform(Image.open(anchor_path).convert('RGB'))
            positive = self.transform(Image.open(pos_path).convert('RGB'))
            negative = self.transform(Image.open(neg_path).convert('RGB'))
            return anchor, positive, negative
        except Exception as e:
            # Si una imagen está corrupta, usar idx+1
            return self.__getitem__((idx + 1) % len(self))

dataset = TripletDataset(pet_groups, train_transform)
loader = DataLoader(dataset, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
print(f'Dataset: {len(dataset)} muestras, {len(loader)} batches de 8')

## 6. Training loop con triplet loss

Solo fine-tuneamos las últimas capas (freeze el backbone) para no romper los embeddings generales y evitar overfitting.

In [ ]:
# Freeze backbone, solo entrena las ultimas N capas
NUM_TRAINABLE_LAYERS = 4  # ultimos 4 transformer blocks

for param in model.parameters():
    param.requires_grad = False

# Habilitar las ultimas capas
for layer in model.encoder.layer[-NUM_TRAINABLE_LAYERS:]:
    for param in layer.parameters():
        param.requires_grad = True

# Y la layernorm final
for param in model.layernorm.parameters():
    param.requires_grad = True

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f'Entrenable: {trainable_params / 1e6:.1f}M de {total_params / 1e6:.1f}M ({100 * trainable_params / total_params:.1f}%)')

In [ ]:
from torch.optim import AdamW
from torch.nn import TripletMarginLoss
from tqdm import tqdm

LR = 1e-5
EPOCHS = 3  # Empezamos con 3, si queda tiempo subimos
MARGIN = 0.2  # Margen para triplet loss

optimizer = AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
triplet_loss = TripletMarginLoss(margin=MARGIN, p=2)

def get_embedding(model, imgs):
    """Extrae embedding [CLS] token, L2-normalized."""
    outputs = model(pixel_values=imgs)
    # DINOv2: pooler_output = CLS token después de layernorm
    emb = outputs.pooler_output if outputs.pooler_output is not None else outputs.last_hidden_state[:, 0, :]
    return F.normalize(emb, p=2, dim=1)

model.train()
for epoch in range(EPOCHS):
    losses = []
    pbar = tqdm(loader, desc=f'Epoch {epoch+1}/{EPOCHS}')
    for anchor, positive, negative in pbar:
        anchor = anchor.to(device)
        positive = positive.to(device)
        negative = negative.to(device)

        emb_a = get_embedding(model, anchor)
        emb_p = get_embedding(model, positive)
        emb_n = get_embedding(model, negative)

        loss = triplet_loss(emb_a, emb_p, emb_n)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        losses.append(loss.item())
        pbar.set_postfix({'loss': f'{sum(losses[-50:]) / min(50, len(losses)):.4f}'})

    avg_loss = sum(losses) / len(losses)
    print(f'Epoch {epoch+1}: avg loss = {avg_loss:.4f}')

## 7. Evaluar sobre tus 49 fotos de test

Comparamos mismo-mascota vs distinta-mascota con el modelo fine-tuneado y comparamos contra baseline 0.445.

In [ ]:
from itertools import combinations
import numpy as np

TEST_DIR = '/content/test_photos'

def cosine_sim(a, b):
    a = a.flatten()
    b = b.flatten()
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))

def extract_embedding_eval(model, img_path):
    img = Image.open(img_path).convert('RGB')
    w, h = img.size
    crop_size = min(w, h)
    left = (w - crop_size) // 2
    top = (h - crop_size) // 2
    img = img.crop((left, top, left + crop_size, top + crop_size))
    tensor = eval_transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        emb = get_embedding(model, tensor)
    return emb.cpu().numpy().squeeze()

# Recolectar fotos
pets_test = {}
for pet_dir in os.listdir(TEST_DIR):
    full_path = os.path.join(TEST_DIR, pet_dir)
    if os.path.isdir(full_path):
        photos = [os.path.join(full_path, f) for f in os.listdir(full_path) 
                  if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))]
        if len(photos) >= 2:
            pets_test[pet_dir] = photos

print(f'Mascotas test: {len(pets_test)}')
print(f'Total fotos: {sum(len(v) for v in pets_test.values())}')

In [ ]:
model.eval()

# Extraer embeddings
embeddings = {}
for pet_name, photos in pets_test.items():
    embeddings[pet_name] = []
    for p in tqdm(photos, desc=pet_name):
        emb = extract_embedding_eval(model, p)
        embeddings[pet_name].append(emb)

# Same-pet y cross-pet
same_sims, cross_sims = [], []

for pet, embs in embeddings.items():
    for i, j in combinations(range(len(embs)), 2):
        same_sims.append(cosine_sim(embs[i], embs[j]))

pets_list = list(embeddings.keys())
for pa, pb in combinations(pets_list, 2):
    for ea in embeddings[pa]:
        for eb in embeddings[pb]:
            cross_sims.append(cosine_sim(ea, eb))

same_mean = np.mean(same_sims)
cross_mean = np.mean(cross_sims)
separation = same_mean - cross_mean

print('\n' + '=' * 60)
print('EVALUACION MODELO FINE-TUNEADO')
print('=' * 60)
print(f'Pares same-pet:  {len(same_sims)}')
print(f'Pares cross-pet: {len(cross_sims)}')
print(f'Same-pet mean:   {same_mean:.4f}')
print(f'Cross-pet mean:  {cross_mean:.4f}')
print(f'Separacion:      {separation:.4f}')
print('=' * 60)
print('COMPARATIVA vs BASELINE (DINOv2-large sin fine-tuning):')
print(f'  Baseline:    same=0.718 cross=0.272 separacion=0.445')
print(f'  Fine-tuned:  same={same_mean:.3f} cross={cross_mean:.3f} separacion={separation:.3f}')
delta = separation - 0.445
print(f'\n  DELTA: {"+" if delta > 0 else ""}{delta:.3f}')
if separation > 0.80:
    print('  [EXCELENTE] Sobrepasamos target 0.80, listo para produccion')
elif separation > 0.65:
    print('  [BUENO] Mejora significativa, seguir iterando (mas epocas)')
elif separation > 0.50:
    print('  [OK] Mejora modesta, probar otros hiperparametros')
else:
    print('  [POBRE] Revisar: mas datos? otra loss? protocolo captura?')

## 8. Guardar modelo + descargar

In [ ]:
OUTPUT_PATH = '/content/dinov2_large_nose_finetuned.pt'

# Guardar solo state_dict (mas chico, ~1.2 GB)
torch.save({
    'model_state_dict': model.state_dict(),
    'model_id': MODEL_ID,
    'num_trainable_layers': NUM_TRAINABLE_LAYERS,
    'epochs': EPOCHS,
    'lr': LR,
    'margin': MARGIN,
    'eval_separation': float(separation),
    'eval_same_mean': float(same_mean),
    'eval_cross_mean': float(cross_mean),
}, OUTPUT_PATH)

import os
size_mb = os.path.getsize(OUTPUT_PATH) / 1024 / 1024
print(f'Modelo guardado: {OUTPUT_PATH} ({size_mb:.0f} MB)')

In [ ]:
# Copiar a Drive (mas facil que download)
DRIVE_DEST = '/content/drive/MyDrive/Paw Friend/dinov2_large_nose_finetuned.pt'
os.makedirs(os.path.dirname(DRIVE_DEST), exist_ok=True)
!cp {OUTPUT_PATH} "{DRIVE_DEST}"
print(f'Copiado a Drive: {DRIVE_DEST}')
print('Se puede bajar desde: https://drive.google.com/')

## 9. Siguiente paso

Pasame a Claude los números finales (`separation`, `same_mean`, `cross_mean`) y decidimos:

- **separación >0.80**: deploy modelo, activar feature nose print en producción
- **0.65–0.80**: iterar con más épocas / diferente learning rate / más data (DogFaceNet real)
- **<0.65**: problema de protocolo de captura, no de modelo — foco en página `/nose-print-test` para generar dataset propio con captura estandarizada

Para deployar el modelo:
- Subir `.pt` a Supabase Storage bucket privado `models/`
- Edge function `nose-print-embed` lo descarga al arrancar
- O exponer vía HuggingFace Inference endpoint (más escalable)